In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS

from scipy import sparse
from xarray import DataArray
from scipy.sparse.linalg import eigsh
import numpy as np

import pyvista as pv
import pyansys

from pyFBS.utility import *


In [33]:
def read_ansys_full_file(full_file_name, result_file_name, nodes_from_matlab = None):
    
    fobj = pyansys.read_binary(full_file_name)
    dof_reference_table, k, m = fobj.load_km(sort=False)  # returns upper triangle only
    ndof_per_node = fobj.ndof[0]
    
    k += sparse.triu(k, 1).T.todense()
    m += sparse.triu(m, 1).T.todense()
    
#    k, m = sort_matrices_from_nodes(k.todense(), m.todense(), dof_reference_table, nodes_from_matlab, ndof_per_node)
        
    nodes_and_coords_from_rst_file = {}
    
    result = pyansys.read_binary(result_file_name)
    coords_from_rst_file = result.geometry["nodes"]
    
    nodes_and_coords_from_rst_file = {key:coords_from_rst_file[key-1][0:3] for key in dof_reference_table[::3][:,0]\
                                       if key not in nodes_and_coords_from_rst_file}
    
    if nodes_from_matlab is not None:
        
        nodes_and_coords_mat_file = {key[1]:nodes_and_coords_from_rst_file[key_1] for key in nodes_from_matlab for key_1 in \
                                     nodes_and_coords_from_rst_file if key[1] == key_1}
        
    else:
        
        nodes_and_coords_mat_file = nodes_and_coords_from_rst_file
        
    
    return k, m, dof_reference_table, ndof_per_node, nodes_and_coords_mat_file

In [34]:
full_file = '../data/AM_automotive_testbench/FEM/EM/EM.full'
result_file = '../data/AM_automotive_testbench/FEM/EM/EM.rst'

k, m, dof_reference_table, ndof_per_node, nodes_and_coordinates = read_ansys_full_file(full_file, result_file)


In [35]:
display(k.shape,m.shape,dof_reference_table.shape,ndof_per_node)

(13404, 13404)

(13404, 13404)

(13404, 2)

3

In [36]:
no_modes = 10
eigen_val, eigen_vec  = eigsh(k, k=no_modes, M=m, sigma = 10000, tol = 1e-3)
omega = [round(np.sqrt(abs(item))) for item in eigen_val]

In [41]:
df = pd.DataFrame.from_dict(nodes_and_coordinates,orient='index')
df_1 = df.sort_index()

df_1

,0,1,2
1,0.394330,0.367819,0.267206
2,0.394330,0.367819,0.262761
3,0.394330,0.367819,0.258317
4,0.394330,0.367819,0.253872
5,0.394330,0.367819,0.249428
...,...,...,...
4464,0.380652,0.354497,0.269428
4465,0.383268,0.350897,0.269428
4466,0.381675,0.352489,0.271650
4467,0.380300,0.356721,0.271650


In [65]:
view3D = pyFBS.display.view3D()

In [66]:
point_cloud = pv.PolyData(df_1.to_numpy()*1000)
mesh_actor = view3D.plot.add_mesh(point_cloud,scalars = np.zeros(point_cloud.points.shape[0]) ,color = "k",name = "mesh")#,show_edges=True)#,render_points_as_spheres = True,point_size=20)

pts = point_cloud.points.copy()

In [74]:
engine_mount = "../data/AM_automotive_testbench/STL/engine_mount.stl"
view3D.add_stl(engine_mount,name = "engine_mount",color = "#8FB1CC",opacity = 0.2)

In [75]:
_modeshape = np.zeros_like(df_1.to_numpy())

select_mode = 6
i = 0
for ref,mode in zip(dof_reference_table,eigen_vec[:,select_mode]):
    _modeshape[ref[0]-1,ref[1]] = mode


In [76]:
mode_dict = dict()

mode_dict["freq"] = 0#a.nat_freq[select]
mode_dict["damp"] = 0
mode_dict["mcf"] = 0

mode_dict["animation_pts"] = mode_animation(_modeshape,10,no_points = 60)
mode_dict["mesh"] = point_cloud
mode_dict["or_pts"] = pts

mode_dict["scalars"] = True
mode_dict["fps"] = 30

view3D.add_modeshape(mode_dict,run_animation = True)

Exception: Number of scalars (4468) must match either the number of points (176614) or the number of cells (300055). 